# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [13]:
from IPython.display import Markdown, display

display(Markdown("""
### Unit of Analysis

- **One row represents:** one content item for one client on one report date.
- **Dataset used:** `fact_content_daily_performance_sample`
- **Time window:** March 2026 (chosen as a representative mid-panel month).

This means each record contains the daily search performance metrics for a single content page belonging to one client on one specific day.
"""))

from huggingface_hub import hf_hub_download
import duckdb

# Download the sample parquet locally
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

con = duckdb.connect()

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{parquet_file}')
""").df()


### Unit of Analysis

- **One row represents:** one content item for one client on one report date.
- **Dataset used:** `fact_content_daily_performance_sample`
- **Time window:** March 2026 (chosen as a representative mid-panel month).

This means each record contains the daily search performance metrics for a single content page belonging to one client on one specific day.


,total_rows,start_date,end_date
0,11694072,2026-06-01,2026-06-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [14]:
from IPython.display import Markdown, display

display(Markdown("""
# Field Classification

## Features
These fields are available before making a prediction.

- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

## Label / Proxy
The target is whether a content page should be refreshed based on its future performance.

## Context
These columns identify records but should never be used as model features.

- client_id
- content_id
- report_date

## Excluded
These columns are excluded because they contain future information, identifiers, or could cause data leakage.
"""))


# Field Classification

## Features
These fields are available before making a prediction.

- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

## Label / Proxy
The target is whether a content page should be refreshed based on its future performance.

## Context
These columns identify records but should never be used as model features.

- client_id
- content_id
- report_date

## Excluded
These columns are excluded because they contain future information, identifiers, or could cause data leakage.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:

from IPython.display import display, Markdown

display(Markdown("# 3. Verification Queries"))

# -----------------------------
# Query 0 : Inspect schema
# -----------------------------
display(Markdown("## Query 0 : Dataset Columns"))

columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{parquet_file}')
""").df()

display(columns)

# -----------------------------
# Query 1 : Row count & window
# -----------------------------
display(Markdown("## Query 1 : Row Count and Date Range"))

display(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{parquet_file}')
    """).df()
)

# -----------------------------
# Query 2 : Grain verification
# -----------------------------
display(Markdown("## Query 2 : Grain Verification"))

display(
    con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_rows
    FROM read_parquet('{parquet_file}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
    """).df()
)

# -----------------------------
# Query 3 : Availability check
# -----------------------------
display(Markdown("## Query 3 : GA4 Availability"))

display(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN 1
                ELSE 0
            END
        ) AS available_rows
    FROM read_parquet('{parquet_file}')
    """).df()
)

# 3. Verification Queries

## Query 0 : Dataset Columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## Query 1 : Row Count and Date Range

,total_rows,start_date,end_date
0,11694072,2026-06-01,2026-06-30


## Query 2 : Grain Verification

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows
0,2026-06-14,client_def0955f7a377868,content_ac604330c75afd04,2
1,2026-06-14,client_4a18d1793d92fb84,content_70d38d7f46fb733a,2
2,2026-06-14,client_e00b29e582949543,content_4b1e89b995d62ac2,2
3,2026-06-20,client_810019792c9b8efc,content_3cb9038dd7c2bb7a,2
4,2026-06-21,client_810019792c9b8efc,content_695b0667a5627edd,2
5,2026-06-21,client_1a730cb2640a1abf,content_c426957b8369a333,2
6,2026-06-17,client_1a8bf67cad4ee525,content_d5fd9414f44c3d2b,2
7,2026-06-17,client_1a8bf67cad4ee525,content_d0e1a961e6d3769a,2
8,2026-06-19,client_8ddc46da5414ffd8,content_409345e43857c17d,2
9,2026-06-20,client_a2eeb8899886adde,content_97a0041b110ebc25,2


## Query 3 : GA4 Availability

,total_rows,available_rows
0,11694072,644726.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [16]:
from IPython.display import Markdown, display

display(Markdown("""
# Data Limitations

### Limitation 1
The sample dataset only contains June 2026, so it cannot be used to study long-term trends or build future-looking labels.

### Limitation 2
Only a subset of rows has GA4 data available, meaning engagement-based features require filtering using `ga4_data_available`.

### Limitation 3
The duplicate check showed repeated combinations of report date, client, and content. This suggests the dataset grain is not perfectly unique and should be verified before model training.

### Limitation 4
Client and content identifiers are pseudonymized (`client_hash_id`, `content_hash_id`), so they should be treated as context fields rather than predictive features.
"""))


# Data Limitations

### Limitation 1
The sample dataset only contains June 2026, so it cannot be used to study long-term trends or build future-looking labels.

### Limitation 2
Only a subset of rows has GA4 data available, meaning engagement-based features require filtering using `ga4_data_available`.

### Limitation 3
The duplicate check showed repeated combinations of report date, client, and content. This suggests the dataset grain is not perfectly unique and should be verified before model training.

### Limitation 4
Client and content identifiers are pseudonymized (`client_hash_id`, `content_hash_id`), so they should be treated as context fields rather than predictive features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [17]:
✅ Every section has a query proving it.
✅ Every feature is knowable at prediction time.
✅ IDs are context, not features.
✅ I named at least one limitation.
✅ Someone else could reproduce my framing.

SyntaxError: invalid character '✅' (U+2705) (2906473496.py, line 1)

In [ ]:
from IPython.display import display, Markdown

display(Markdown("# Five Feature Frame"))

feature_query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{parquet_file}')
WHERE ga4_data_available IS TRUE
LIMIT 10
"""

feature_df = con.sql(feature_query).df()

display(feature_df)

# Five Feature Frame

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,1,1,9.000000,1,0
1,3,1,5.333333,1,0
2,2,1,2.000000,1,0
3,2,1,5.000000,1,0
4,15,0,11.933333,1,0
5,2,0,4.000000,1,0
6,2,1,6.000000,1,0
7,1,1,0.000000,1,0
8,1,0,2.000000,2,0
9,26,2,3.038462,2,0


In [ ]:
display(Markdown("""
## Why these features are safe

| Feature | Knowable at the decision moment because... |
|----------|--------------------------------------------|
| gsc_impressions | Historical Google Search impressions already exist before making a refresh decision. |
| gsc_clicks | Historical clicks are available before prediction. |
| gsc_avg_position | Average search position is known from past Search Console data. |
| ga4_sessions | Previous user sessions are historical engagement data. |
| ga4_engaged_sessions | Previous engaged sessions are historical and available before prediction. |
"""))


## Why these features are safe

| Feature | Knowable at the decision moment because... |
|----------|--------------------------------------------|
| gsc_impressions | Historical Google Search impressions already exist before making a refresh decision. |
| gsc_clicks | Historical clicks are available before prediction. |
| gsc_avg_position | Average search position is known from past Search Console data. |
| ga4_sessions | Previous user sessions are historical engagement data. |
| ga4_engaged_sessions | Previous engaged sessions are historical and available before prediction. |


In [ ]:
display(Markdown("""
# Leakage Demonstration

Suppose we create a label indicating whether a page receives high future clicks.

If we accidentally include that same future-click information as a feature, the model would achieve an unrealistically high score because it is using information from the future.

This is called **data leakage**.

Therefore, future-derived columns must never be used as model features.
"""))


# Leakage Demonstration

Suppose we create a label indicating whether a page receives high future clicks.

If we accidentally include that same future-click information as a feature, the model would achieve an unrealistically high score because it is using information from the future.

This is called **data leakage**.

Therefore, future-derived columns must never be used as model features.


In [ ]:
display(Markdown("""
# Limitation

This notebook uses the June 2026 sample dataset only.

It does not represent the complete 17-month warehouse history.

The sample is suitable for understanding the schema and developing queries, but it should not be used for building production models or evaluating future prediction performance.
"""))


# Limitation

This notebook uses the June 2026 sample dataset only.

It does not represent the complete 17-month warehouse history.

The sample is suitable for understanding the schema and developing queries, but it should not be used for building production models or evaluating future prediction performance.
